## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [2]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [ ]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


IndentationError: unexpected indent (1245075795.py, line 3)

In [4]:
# Here is the final output

print(result.final_output)

Why did the Autonomous AI Agent bring a ladder to work?

Because it wanted to *take initiative*… and then immediately asked another agent to verify the ladder was the right height.


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_01ba2e50d1e05b61006a82b84d09bc819685658f8d83eab54d',
  'content': [{'annotations': [],
    'text': 'Why did the Autonomous AI Agent bring a ladder to work?\n\nBecause it wanted to *take initiative*… and then immediately asked another agent to verify the ladder was the right height.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

#set_tracing_disabled(True)
#agents.run.RunConfig.tracing.disabled = True

Why did the Autonomous AI Agent start a band?

Because it wanted to **take initiative, learn from feedback, and keep looping until it found the right “note”**.


## Now go and look at the trace

https://platform.openai.com/traces

In [8]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Absolutely — here are 5 AI agent jokes:

1. **Why did the AI agent get promoted?**  
   Because it always *followed the prompt* and never missed a deadline.

2. **What do AI agents say when they’re confused?**  
   “Let me think step by step… and then step sideways.”

3. **Why was the AI agent bad at keeping secrets?**  
   It kept giving away its *thought process*.

4. **How do AI agents stay in shape?**  
   They do lots of **prompt-cise** training.

5. **Why did the AI agent bring a ladder to work?**  
   It wanted to reach a higher **context window**.

If you want, I can also do:
- more **nerdy AI jokes**
- **short one-liners**
- **dad jokes**
- or **slightly darker machine-learning humor**

## Part 2: Adding a tool

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")


TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

Pushover user not found
Pushover token not found


In [4]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


def send_telegram_message(text):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": text}

    response = requests.post(url, data=payload)

    if response.status_code == 200:
        # print("Message sent successfully!")
        return {"status": "success", "message": text}
    else:
        # print(f"Failed to send message. Status code: {response.status_code}")
        # print(response.text)
        return {"status": "error", "message": response.text}

In [5]:
push("HEY!!")
send_telegram_message("Hey there!")

Push: HEY!!


{'status': 'success', 'message': 'Hey there!'}

In [7]:
push 

<function __main__.push(message)>

In [9]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

@function_tool
def send_tool(text: str) -> str: 
    """ Send the given text to the user as a telegram message. """
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": text}

    response = requests.post(url, data=payload)

    if response.status_code == 200:
        return {"status": "success", "message": text}
    else:
        return {"status": "error", "message": response.text}

In [11]:
push_tool

send_tool

FunctionTool(name='send_tool', description='Send the given text to the user as a telegram message.', params_json_schema={'properties': {'text': {'title': 'Text', 'type': 'string'}}, 'required': ['text'], 'title': 'send_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x1146fe990>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [13]:
push_tool.description

send_tool.description

'Send the given text to the user as a telegram message.'

In [14]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[send_tool])

In [16]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Notified the user: “The pizza is here.”


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [17]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [18]:
response = await Runner.run(agent, "Hi there. My name is Laura.")
print(response.final_output)

Hi Laura — nice to meet you. How can I help today?


In [19]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me.


## Memory approach 1 - just manually pass in the list of dicts

In [20]:
response = await Runner.run(agent, "Hi there. My name is Laura.")
print(response.final_output)

Hi Laura — nice to meet you! How can I help you today?


In [21]:
response.to_input_list()

[{'content': 'Hi there. My name is Laura.', 'role': 'user'},
 {'id': 'msg_0aa3d867759c714d006a83fd903ab087d1b50d36c9e18e768e',
  'content': [{'annotations': [],
    'text': 'Hi Laura — nice to meet you! How can I help you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [22]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Laura.', 'role': 'user'},
 {'id': 'msg_0aa3d867759c714d006a83fd903ab087d1b50d36c9e18e768e',
  'content': [{'annotations': [],
    'text': 'Hi Laura — nice to meet you! How can I help you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [23]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Laura.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [24]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [25]:
response = await Runner.run(agent, "Hi there. My name is Laura.", session=session)
print(response.final_output)

Hi Laura — nice to meet you! How can I help you today?


In [26]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Laura.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>